# Programming for Data Science — Mini Session 04
## Assignment: More pandas & SQLite
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cgpan/humd5001_mini/blob/main/assignment/04_Pandas_SQLite_Assignment.ipynb)

[**Chenguang Pan**](https://cpan.ai/), [**Youmi Lab**](https://youmilab.ai/)
**Teachers College, Columbia University**

This assignment goes with the in-class notebook **`04_Pandas_SQLite.ipynb`**. In class we worked with the Titanic passengers; here you get a fresh dataset and a good deal more independence.

---
### 📋 How this assignment works

- **Run on Google Colab.** Open this notebook with the badge above, then *File ▸ Save a copy in Drive* so you have your own copy to edit.
- **The data.** The **Palmer penguins**: 344 birds from three species on three Antarctic islands, with bill, flipper, and body-mass measurements. It is a modern classic — and, like all real data, it has gaps you will need to deal with.
- **Scope.** Everything builds on **Sessions 01–04**. You still will not need `for` loops, `if` statements, or your own functions (`def`) — and please do not use them. pandas and SQL are both *declarative*: you describe the result you want and let them do the looping.
- **Three parts:**
  1. **Part A — Multiple Choice** (10 questions): check your understanding.
  2. **Part B — Basic Exercises** (6 questions): short drills, one Session-04 idea each.
  3. **Part C — Integrated Challenges** (4 questions): full analyses that reach back through **every session so far** — strings and dictionaries from 01, NumPy and JSON from 02, pandas from 03, joins and SQL from 04.
- **Difficulty** runs from ★☆☆☆☆ (warm-up) to ★★★★★ (make-you-think). The **Skills** line names the tools each question is really about.
- Use **clear variable names** and drop a **comment** on the tricky lines. Six months from now, that comment is a gift to yourself.

**Total: 100 points** — Part A: 20 · Part B: 30 · Part C: 50.

---
### ✍️ Before you start
Put your name in the next cell, then run the setup cell right after it.

In [ ]:
student_name = ""   # <-- type your name between the quotes
print("Submission by:", student_name)

In [ ]:
# === SETUP — run me first! ===
import numpy as np
import pandas as pd
import sqlite3
import json, os

# Both tables live in the public course repo, so they load straight from the web.
PENGUINS_URL = "https://raw.githubusercontent.com/cgpan/humd5001_mini/main/assets/penguins.csv"
SPECIES_URL  = "https://raw.githubusercontent.com/cgpan/humd5001_mini/main/assets/penguin_species.csv"

penguins = pd.read_csv(PENGUINS_URL)
species_info = pd.read_csv(SPECIES_URL)

print("penguins:", penguins.shape, "| species_info:", species_info.shape)
print("pandas", pd.__version__, "| sqlite3", sqlite3.sqlite_version)
penguins.head()

---
## Part A — Multiple Choice  *(2 points each, 20 total)*

Reason each one out — you should not need to run any code for Part A. When you are done, record your letters in the answer dictionary at the end of this part.

#### Question A1
**Difficulty:** ★★☆☆☆  **Skills:** `concat` vs `merge`

You have two tables with the **same columns** — this month's records and last month's — and you want one longer table containing all the rows. Which tool?

- **A.** `pd.concat([a, b])`
- **B.** `pd.merge(a, b, on="id")`
- **C.** `pd.pivot_table(a, b)`
- **D.** `a.corr(b)`

#### Question A2
**Difficulty:** ★★★☆☆  **Skills:** `merge(how=...)`

`pd.merge(left, right, on="key", how="left")` is used. A row in `left` has a key that appears **nowhere** in `right`. What happens to it?

- **A.** It is dropped from the result.
- **B.** It raises a `KeyError`.
- **C.** It is kept, with `NaN` in the columns that came from `right`.
- **D.** It is filled with zeros.

#### Question A3
**Difficulty:** ★★★☆☆  **Skills:** skew, `median` vs `mean`

A `fare` column has mean 32, median 14, and max 512. Which statement is best supported?

- **A.** The data are symmetric; either statistic describes them well.
- **B.** The mean is wrong and should never be reported.
- **C.** The column is right-skewed, so the **median** better describes a typical value.
- **D.** Half the values are above 32.

#### Question A4
**Difficulty:** ★★☆☆☆  **Skills:** `.isna().sum()`

What does `df.isna().sum()` return?

- **A.** the total number of rows
- **B.** a table with the missing rows removed
- **C.** `True` if any value is missing
- **D.** the number of missing values **in each column**

#### Question A5
**Difficulty:** ★★★☆☆  **Skills:** `.dropna()` vs `.fillna()`

Your `sex` column has 11 missing values out of 344 rows. You run `df = df.dropna(subset=["sex"])`. What is the trade-off?

- **A.** You lose those rows entirely, including the measurements they had in **other** columns.
- **B.** No trade-off; this is always the correct choice.
- **C.** It invents data that was never measured.
- **D.** It changes the column's dtype to text.

#### Question A6
**Difficulty:** ★★☆☆☆  **Skills:** `pivot_table`

What does `pd.pivot_table(df, values="mass", index="species", columns="sex", aggfunc="mean")` produce?

- **A.** one mean mass per species
- **B.** a grid of mean mass with species down the side and sex across the top
- **C.** the raw rows sorted by species, then by sex
- **D.** the total mass of all penguins

#### Question A7
**Difficulty:** ★☆☆☆☆  **Skills:** SQL `SELECT` / `WHERE`

Which SQL statement returns the `species` and `body_mass_g` of penguins heavier than 5000 g?

- **A.** `SELECT * FROM penguins GROUP BY body_mass_g > 5000`
- **B.** `SELECT species, body_mass_g FROM penguins WHERE body_mass_g > 5000`
- **C.** `FROM penguins SELECT species WHERE body_mass_g > 5000`
- **D.** `SELECT species, body_mass_g WHERE body_mass_g > 5000`

#### Question A8
**Difficulty:** ★★★☆☆  **Skills:** SQL `GROUP BY`

Which query is the SQL equivalent of `df.groupby("species")["body_mass_g"].mean()`?

- **A.** `SELECT species, AVG(body_mass_g) FROM penguins`
- **B.** `SELECT species, MEAN(body_mass_g) FROM penguins GROUP BY species`
- **C.** `SELECT AVG(species) FROM penguins GROUP BY body_mass_g`
- **D.** `SELECT species, AVG(body_mass_g) FROM penguins GROUP BY species`

#### Question A9
**Difficulty:** ★★☆☆☆  **Skills:** `pd.read_sql_query`

What does `pd.read_sql_query("SELECT * FROM penguins", conn)` return?

- **A.** a list of tuples
- **B.** a cursor object
- **C.** a pandas **DataFrame**
- **D.** a string

#### Question A10
**Difficulty:** ★★★★☆  **Skills:** SQL `JOIN` ↔ `merge`

In SQL, `FROM penguins AS p LEFT JOIN species AS s ON p.species = s.species` is the counterpart of which pandas call?

- **A.** `pd.concat([penguins, species])`
- **B.** `penguins.groupby("species").mean()`
- **C.** `penguins.sort_values("species")`
- **D.** `pd.merge(penguins, species, on="species", how="left")`

#### ✅ Record your Part A answers
Fill in the dictionary below so each number points to your chosen letter (for example, `1: "C"`).

In [ ]:
my_mcq_answers = {
    1 : "",   # Question A1
    2 : "",   # Question A2
    3 : "",   # Question A3
    4 : "",   # Question A4
    5 : "",   # Question A5
    6 : "",   # Question A6
    7 : "",   # Question A7
    8 : "",   # Question A8
    9 : "",   # Question A9
    10: "",   # Question A10
}
print(my_mcq_answers)

---
## Part B — Basic Exercises  *(5 points each, 30 total)*

Each drill zooms in on **one** idea from Session 04. Write your answer in the empty cell under each prompt. Short is fine — focused is the goal.

#### Exercise B1 — A descriptive profile
**Difficulty:** ★★☆☆☆  **Points:** 5
**Skills:** `.describe()`, `.agg()`, `median` vs `mean`

*(Uses the `penguins` DataFrame loaded in the setup cell.)*

1. Print `.describe()` for `body_mass_g`, rounded to 1 decimal.
2. Using **one** `.agg()` call, print the `count`, `mean`, `median`, and `std` of `body_mass_g`.
3. In a comment, say whether the mean and median are close, and what that suggests about the shape of the distribution.

In [ ]:
# Your code here


#### Exercise B2 — Auditing the gaps
**Difficulty:** ★★★☆☆  **Points:** 5
**Skills:** `.isna().sum()`, `.dropna()`, `.shape`

Real data has holes, and this dataset is no exception.

1. Print the number of missing values **in each column** of `penguins`.
2. Make a copy with every incomplete row removed (`.dropna()`), and print the row count **before** and **after**.
3. Print how many rows were lost (a subtraction).

In [ ]:
# Your code here


#### Exercise B3 — Tidying the table
**Difficulty:** ★★☆☆☆  **Points:** 5
**Skills:** `.rename()`, `.drop(columns=)`, `.duplicated()`

Work on a **copy** of `penguins` so the original stays intact.

1. Rename `bill_length_mm` to `bill_len` and `bill_depth_mm` to `bill_depth`.
2. Drop the `bill_depth` column and print the remaining column names as a list.
3. Print how many exact **duplicate rows** the original `penguins` table contains.

In [ ]:
# Your code here


#### Exercise B4 — Statistics by group
**Difficulty:** ★★★☆☆  **Points:** 5
**Skills:** `groupby`, `.agg()`

1. Group `penguins` by `species` and report the `count`, `mean`, and `std` of `body_mass_g` in one `.agg()` call, rounded to 1 decimal.
2. Print the average `flipper_length_mm` for each `island`, rounded to 1 decimal.

In [ ]:
# Your code here


#### Exercise B5 — Stacking and joining
**Difficulty:** ★★★☆☆  **Points:** 5
**Skills:** `pd.concat`, `pd.merge`, `how="left"`, `.shape`

Two different jobs, side by side. The `species_info` table (loaded in setup) maps each species to its scientific name.

1. **Stack:** combine the first 3 rows and the last 3 rows of `penguins` (`.head(3)` and `.tail(3)`) with `pd.concat`, and print the resulting shape — you should get 6 rows.
2. **Join:** left-join `species_info` onto the full `penguins` table using `on="species"`, storing the result as `merged`.
3. Print the shape **before** and **after** the join (the row count should not change), then show the `species` and `scientific_name` columns for the first 3 rows.

In [ ]:
# Your code here


#### Exercise B6 — Building a table by hand
**Difficulty:** ★★★★☆  **Points:** 5
**Skills:** `sqlite3.connect`, `cursor`, `CREATE TABLE`, `executemany`, `commit`, `fetchall`

In class you saw two ways into a database: the one-line `to_sql`, and the manual route with a cursor. Here you take the manual route, so the machinery is not a mystery.

1. Connect to a database file called `penguins.db` and create a **cursor**.
2. Run `DROP TABLE IF EXISTS islands` (this keeps the cell safe to re-run), then `CREATE TABLE` an `islands` table with two TEXT columns: `island` and `research_base`.
3. Insert these three rows with **one** `executemany` call, then `commit`:

```python
rows = [
    ("Torgersen", "Palmer Station"),
    ("Biscoe", "Palmer Station"),
    ("Dream", "Palmer Station"),
]
```

4. Read them back with `cur.execute("SELECT * FROM islands").fetchall()`, print the list, and then print just the **first island name** using indexing.
5. Close the connection.

In [ ]:
# Your code here


---
## Part C — Integrated Challenges  *(50 points total)*

These are the real thing: complete analyses that pull together **everything from Sessions 01 through 04**. Take them one numbered step at a time, print as you go, and check that each result makes sense before moving on. Still no `for`, `if`, or `def`.

#### Exercise C1 — Cleaning & profiling — an honest first report
**Difficulty:** ★★★★☆  **Points:** 12
**Skills:** `.isna()`, `.fillna()`, `.median()`, `.agg()`, `value_counts()`, `len(set(...))`, f-string

**The setting.** Before anyone trusts your analysis, you have to say what shape the data arrived in and what you did about it. This is the audit step — unglamorous, and the difference between a real analysis and a plausible-looking one.

1. Print the missing-value count for each column of `penguins`.
2. Build `clean` — a copy where the missing **`body_mass_g`** values are filled with the column's **median**. Confirm with `.isna().sum()` that `body_mass_g` has no gaps left.
3. On `clean`, print `count`, `mean`, `median`, and `std` for `body_mass_g` in one `.agg()` call (round to 1).
4. Print the `value_counts()` of `species` **and** of `island`.
5. Using `normalize=True`, print the **proportion** of penguins of each species, rounded to 3.
6. Finish with an f-string that reports what you did, for example: `Filled 2 missing body_mass values with the median (4050.0). 344 penguins across 3 species.`

In [ ]:
# Your code here


#### Exercise C2 — Comparative statistics — who is the biggest?
**Difficulty:** ★★★★☆  **Points:** 12
**Skills:** `groupby` (two keys), `.agg()`, `pivot_table`, `pd.merge`, `.corr()`

**The setting.** The dataset exists to answer a comparative question: do the three species differ, and how? This is descriptive statistics doing real work — the same moves you would make before any modelling.

1. Group by `species` and report `count`, `mean`, and `std` of `flipper_length_mm` (round to 1).
2. Group by **both** `species` and `sex` and print the mean `body_mass_g` (round to 1).
3. Turn that same two-key summary into a readable grid with `pivot_table`: `body_mass_g` as the values, `species` down the side, `sex` across the top, `aggfunc="mean"`, rounded to 1.
   Then check how many penguins that grid actually covers: group by `["species", "sex"]`, count the `penguin_id` column, and `.sum()` those group counts. Compare the total with `penguins.shape[0]` and, in a comment, say where the missing birds went.
4. Merge in `species_info` (left join on `species`), then print the mean `body_mass_g` grouped by **`scientific_name`**, rounded to 1.
5. Print the **correlation** between `flipper_length_mm` and `body_mass_g`, rounded to 3. In a comment, say in one sentence what that number means.

In [ ]:
# Your code here


#### Exercise C3 — SQLite end-to-end — the same questions, twice
**Difficulty:** ★★★★★  **Points:** 13
**Skills:** `sqlite3`, `to_sql`, `pd.read_sql_query`, `GROUP BY`, `ORDER BY`, `LIMIT`, `LEFT JOIN`, `groupby` + `.agg()`

**The setting.** The habit from class: ask a question in SQL, then ask it again in pandas, and check that the two answers agree. That is how you build trust in a new language — and how you catch your own mistakes. In **B6** you built a table by hand; here you take the fast route with `to_sql`.

1. Connect to `penguins.db`. Write **both** tables into it: `penguins` as `birds`, and `species_info` as `species_info` (use `index=False, if_exists="replace"` for both).
2. **SQL:** return `species`, `island`, and `body_mass_g` for the **5 heaviest** penguins (`ORDER BY ... DESC`, `LIMIT 5`).
3. **SQL:** for each `species`, return the number of penguins and the average `body_mass_g` rounded to 1 (`COUNT(*)`, `ROUND(AVG(...), 1)`, `GROUP BY`). Give the computed columns readable names with `AS`.
4. **pandas:** compute that same species summary with `groupby` + `.agg(["count", "mean"])`. The **average masses will agree exactly** — but two of the **counts will not**. In a comment, explain why. *(Hint: what does `COUNT(*)` count, and what does pandas' `count` skip?)*
5. **SQL:** `LEFT JOIN` `birds` to `species_info` on `species`, and return `scientific_name` with the average `body_mass_g` per species.
6. Close the connection.

In [ ]:
# Your code here


#### Exercise C4 — Capstone — everything you have learned, in one report
**Difficulty:** ★★★★★  **Points:** 13
**Skills:** `str` methods & f-strings (S01), `set`/`dict` (S01), NumPy + boolean indexing (S02), `json`/`os` (S02), `merge`/`groupby` (S03–04), SQL (S04)

**The setting.** One last pass that touches every session. This is what a small, complete analysis actually looks like, end to end — and the tag on each step tells you where that tool came from.

1. **(S03/04)** Make `clean` by dropping rows with a missing `body_mass_g`, then **left-join** `species_info` onto it. Print the resulting shape.
2. **(S02)** Pull the `body_mass_g` column into a **NumPy array** with `np.array(...)`. Using **boolean indexing**, print how many penguins weigh more than **4500 g** (`.sum()` on the mask), and the **mean mass of just those** heavy penguins, rounded to 1.
3. **(S01)** Count the distinct species with `len(set(...))`, and build a **dictionary** mapping each species name to its mean body mass, rounded to 1. *(Hint: `clean.groupby("species")["body_mass_g"].mean().round(1).to_dict()` does this in one line.)*
4. **(S01)** Take the scientific name of the heaviest species and print it in **UPPERCASE**, plus its **first word** (the genus) using `.split()` and indexing.
5. **(S04)** Re-connect to `penguins.db` and write `clean` into it as `clean_birds` (`index=False, if_exists="replace"`), then run **one SQL query** returning each `island` with its penguin count and average body mass, ordered by average mass descending. Close the connection.
6. **(S02)** Assemble a summary **dictionary** (number of penguins, number of species, overall mean mass, count over 4500 g), write it to `penguin_report.json` with `json.dump`, read it back with `json.load`, and confirm the file exists with `os.path.exists`.
   > ⚠️ pandas and NumPy hand back `numpy` numbers, which `json.dump` refuses to write. Wrap each value in `int(...)` or `float(...)` first — see the gotcha note at the end of the class notebook.
7. **(S01)** Close with an f-string, for example: `Analyzed 342 penguins across 3 species; 115 weigh over 4500 g.`

In [ ]:
# Your code here


---
## 📤 Submission

1. In Colab, run *Runtime ▸ Run all* and make sure every cell finishes **without errors**.
2. Double-check that your Part A answer dictionary is filled in.
3. Save or download your notebook (*File ▸ Download ▸ .ipynb*, or share your Colab link) and turn it in on **Canvas** as instructed.

> 💡 Stuck? Re-open `04_Pandas_SQLite.ipynb` — especially the **pandas ↔ SQL** table at the end. Find the pandas line you already know, then read across.